In [1]:
# ============================================================
# KARTZONE INDIA — DELIVERY TABLE CLEANING
# FINAL CORRECTED PIPELINE
# ============================================================

import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

print("=" * 65)
print("  KARTZONE INDIA — DELIVERY DATA CLEANING")
print("=" * 65)

# ============================================================
# STEP 1 — LOAD RAW DELIVERY DATA
# ============================================================

deliveries = pd.read_csv('KartZone_Deliveries.csv')

print(f"\nRaw rows loaded: {len(deliveries):,}")

# ============================================================
# STEP 2 — REMOVE STRUCTURAL DUPLICATES
# ============================================================

deliveries = deliveries.dropna(how='all')
deliveries = deliveries.drop_duplicates()

# One delivery per Delivery_ID
deliveries = deliveries.drop_duplicates(
    subset=['Delivery_ID'],
    keep='first'
)

# One delivery per Order_ID
deliveries = deliveries.drop_duplicates(
    subset=['Order_ID'],
    keep='first'
)

deliveries = deliveries.reset_index(drop=True)

print(f"Rows after structural cleaning: {len(deliveries):,}")

# ============================================================
# STEP 3 — CLEAN DELIVERY_ID / ORDER_ID
# ============================================================

deliveries['Delivery_ID'] = (
    deliveries['Delivery_ID']
    .astype(str)
    .str.strip()
)

deliveries['Order_ID'] = (
    deliveries['Order_ID']
    .astype(str)
    .str.strip()
)

valid_order_format = deliveries['Order_ID'].str.match(
    r'^ORD-KZ-\d+$',
    na=False
)

print(f"Invalid Order_ID format: {(~valid_order_format).sum()}")

deliveries = deliveries[
    valid_order_format
].reset_index(drop=True)

# ============================================================
# STEP 4 — FOREIGN KEY VALIDATION
# ============================================================

orders_clean = pd.read_csv('KartZone_Orders_Final.csv')

valid_order_ids = set(
    orders_clean['Order_ID'].dropna()
)

invalid_fk = ~deliveries['Order_ID'].isin(valid_order_ids)

print(f"Invalid Order_IDs: {invalid_fk.sum()}")

deliveries = deliveries[
    ~invalid_fk
].reset_index(drop=True)

print(f"Rows after foreign-key validation: {len(deliveries):,}")

# ============================================================
# STEP 5 — STANDARDIZE DELIVERY PARTNER
# ============================================================

def standardize_partner(val):

    if pd.isnull(val):
        return 'Unknown'

    val = str(val).strip().lower()

    if 'delhivery' in val or val == 'dlv':
        return 'Delhivery'

    if (
        'blue dart' in val or
        'bluedart' in val or
        'blue-dart' in val
    ):
        return 'Blue Dart'

    if 'dtdc' in val or 'd.t.d.c' in val:
        return 'DTDC'

    if 'ekart' in val:
        return 'Ekart'

    if (
        'xpressbees' in val or
        'xpress bees' in val or
        val == 'xb'
    ):
        return 'XpressBees'

    if 'amazon' in val or val == 'amzn':
        return 'Amazon Logistics'

    return 'Unknown'


deliveries['Delivery_Partner'] = (
    deliveries['Delivery_Partner']
    .apply(standardize_partner)
)

print("\nDelivery Partner:")
print(deliveries['Delivery_Partner'].value_counts())

# ============================================================
# STEP 6 — STANDARDIZE WAREHOUSE CITY
# ============================================================

def standardize_city(val):

    if pd.isnull(val):
        return np.nan

    val = str(val).strip().lower()

    city_map = {
        'mumbai': 'Mumbai Hub',
        'delhi': 'Delhi Hub',
        'bangalore': 'Bangalore Hub',
        'bengaluru': 'Bangalore Hub',
        'chennai': 'Chennai Hub',
        'hyderabad': 'Hyderabad Hub',
        'pune': 'Pune Hub'
    }

    for key, standard in city_map.items():

        if key in val:
            return standard

    return np.nan


deliveries['Warehouse_City'] = (
    deliveries['Warehouse_City']
    .apply(standardize_city)
)

# Fill missing city using mode
deliveries['Warehouse_City'] = (
    deliveries['Warehouse_City']
    .fillna(deliveries['Warehouse_City'].mode()[0])
)

print("\nWarehouse City:")
print(deliveries['Warehouse_City'].value_counts())

# ============================================================
# STEP 7 — PARSE DATE COLUMNS
# ============================================================

def parse_date(val):

    if pd.isnull(val):
        return pd.NaT

    val = str(val).strip()

    if val.lower() in [
        '', 'na', 'n/a', 'nan', 'null', 'none', '-'
    ]:
        return pd.NaT

    formats = [
        '%Y-%m-%d',
        '%d/%m/%Y',
        '%d-%m-%Y',
        '%m/%d/%Y',
        '%d.%m.%Y',
        '%Y/%m/%d'
    ]

    for fmt in formats:

        try:
            return pd.to_datetime(
                datetime.strptime(val, fmt)
            )

        except:
            continue

    return pd.NaT


for col in [
    'Pickup_Date',
    'Actual_Delivery_Date'
]:

    deliveries[col] = deliveries[col].apply(parse_date)

print("\nDate types:")
print(
    deliveries[
        ['Pickup_Date', 'Actual_Delivery_Date']
    ].dtypes
)

# ============================================================
# STEP 8 — EXPECTED DELIVERY DAYS
# ============================================================

deliveries['Expected_Days'] = pd.to_numeric(
    deliveries['Expected_Days'],
    errors='coerce'
)

deliveries['Expected_Days'] = deliveries[
    'Expected_Days'
].where(
    deliveries['Expected_Days'].between(1, 14),
    np.nan
)

deliveries['Expected_Days'] = (
    deliveries['Expected_Days']
    .fillna(
        deliveries['Expected_Days'].median()
    )
    .round()
    .astype(int)
)

print("\nExpected_Days:")
print(deliveries['Expected_Days'].describe())

# ============================================================
# STEP 9 — ACTUAL DELIVERY DAYS
# ============================================================

deliveries['Actual_Delivery_Days'] = pd.to_numeric(
    deliveries['Actual_Delivery_Days'],
    errors='coerce'
)

# Valid range
deliveries['Actual_Delivery_Days'] = deliveries[
    'Actual_Delivery_Days'
].where(
    deliveries['Actual_Delivery_Days'].between(1, 30),
    np.nan
)

print(
    f"\nMissing Actual_Delivery_Days before fill: "
    f"{deliveries['Actual_Delivery_Days'].isnull().sum()}"
)

# Partner-specific realistic average delay
partner_avg_delay = {
    'Delhivery': 1,
    'Blue Dart': 1,
    'DTDC': 4,
    'Ekart': 2,
    'XpressBees': 3,
    'Amazon Logistics': 1,
    'Unknown': 2
}


def fill_actual_days(row):

    if pd.notnull(row['Actual_Delivery_Days']):
        return row['Actual_Delivery_Days']

    delay = partner_avg_delay.get(
        row['Delivery_Partner'],
        2
    )

    return row['Expected_Days'] + delay


deliveries['Actual_Delivery_Days'] = (
    deliveries
    .apply(fill_actual_days, axis=1)
    .round()
    .astype(int)
)

print(
    f"Missing Actual_Delivery_Days after fill: "
    f"{deliveries['Actual_Delivery_Days'].isnull().sum()}"
)

# ============================================================
# STEP 10 — RECALCULATE DELAY DAYS
# ============================================================

deliveries['Delay_Days'] = (
    deliveries['Actual_Delivery_Days']
    - deliveries['Expected_Days']
)

# Keep reasonable early delivery values
deliveries['Delay_Days'] = (
    deliveries['Delay_Days']
    .clip(lower=-3)
)

print("\nDelay_Days:")
print(deliveries['Delay_Days'].describe())

# ============================================================
# STEP 11 — STANDARDIZE DELIVERY STATUS
# ============================================================

def standardize_status(val):

    if pd.isnull(val):
        return np.nan

    val = str(val).strip().lower()

    if val in [
        'delivered',
        'deliverd',
        'complete',
        'completed'
    ]:
        return 'Delivered'

    if (
        'out for' in val or
        val == 'ofd'
    ):
        return 'Out for Delivery'

    if (
        'transit' in val or
        'intransit' in val
    ):
        return 'In Transit'

    if 'fail' in val:
        return 'Failed'

    if (
        'rto' in val or
        'returned to origin' in val or
        'return to origin' in val
    ):
        return 'Returned to Origin'

    if (
        'delay' in val or
        'behind' in val
    ):
        return 'Delayed'

    return np.nan


deliveries['Delivery_Status'] = (
    deliveries['Delivery_Status']
    .apply(standardize_status)
)

# Derive missing statuses from delivery performance
def derive_status(row):

    if pd.notnull(row['Delivery_Status']):
        return row['Delivery_Status']

    delay = row['Delay_Days']

    if delay <= 0:
        return 'Delivered'

    elif delay <= 2:
        return 'In Transit'

    elif delay <= 5:
        return 'Delayed'

    else:
        return 'Delayed'


deliveries['Delivery_Status'] = (
    deliveries
    .apply(derive_status, axis=1)
)

print("\nDelivery Status:")
print(deliveries['Delivery_Status'].value_counts())

# ============================================================
# STEP 12 — DELIVERY ATTEMPTS
# ============================================================

deliveries['Delivery_Attempts'] = pd.to_numeric(
    deliveries['Delivery_Attempts'],
    errors='coerce'
)

deliveries['Delivery_Attempts'] = (
    deliveries['Delivery_Attempts']
    .where(
        deliveries['Delivery_Attempts'].between(1, 5),
        np.nan
    )
)

# Failed deliveries need at least one attempt
deliveries.loc[
    (
        deliveries['Delivery_Status'] == 'Failed'
    ) &
    (
        deliveries['Delivery_Attempts'].isnull()
    ),
    'Delivery_Attempts'
] = 1

# Remaining missing attempts
deliveries['Delivery_Attempts'] = (
    deliveries['Delivery_Attempts']
    .fillna(1)
    .astype(int)
)

print("\nDelivery Attempts:")
print(deliveries['Delivery_Attempts'].value_counts())

# ============================================================
# STEP 13 — DELIVERY COST
# IMPORTANT: KEEP DECIMAL POINT
# ============================================================

def clean_delivery_cost(val):

    if pd.isnull(val):
        return np.nan

    val = str(val).strip()

    # Remove currency labels but KEEP decimal point
    val = re.sub(
        r'₹|Rs\.?|INR|,|\s',
        '',
        val,
        flags=re.IGNORECASE
    )

    try:

        value = float(val)

        if value <= 0:
            return np.nan

        return value

    except:
        return np.nan


deliveries['Delivery_Cost'] = (
    deliveries['Delivery_Cost']
    .apply(clean_delivery_cost)
)

# Fill missing costs by partner median
partner_cost_median = (
    deliveries
    .groupby('Delivery_Partner')['Delivery_Cost']
    .transform('median')
)

deliveries['Delivery_Cost'] = (
    deliveries['Delivery_Cost']
    .fillna(partner_cost_median)
)

# Final fallback
deliveries['Delivery_Cost'] = (
    deliveries['Delivery_Cost']
    .fillna(
        deliveries['Delivery_Cost'].median()
    )
)

deliveries['Delivery_Cost'] = (
    deliveries['Delivery_Cost']
    .round(2)
)

print("\nDelivery Cost:")
print(deliveries['Delivery_Cost'].describe())

# ============================================================
# STEP 14 — CUSTOMER RATING
# ============================================================

deliveries['Customer_Rating'] = pd.to_numeric(
    deliveries['Customer_Rating'],
    errors='coerce'
)

# Valid rating = 1 to 5
deliveries['Customer_Rating'] = (
    deliveries['Customer_Rating']
    .where(
        deliveries['Customer_Rating'].between(1, 5),
        np.nan
    )
)

status_rating_map = {
    'Delivered': 4.0,
    'Out for Delivery': 3.5,
    'In Transit': 3.0,
    'Delayed': 2.5,
    'Failed': 1.5,
    'Returned to Origin': 2.0
}

deliveries['Customer_Rating'] = (
    deliveries
    .apply(
        lambda row:
        status_rating_map.get(
            row['Delivery_Status'],
            3.0
        )
        if pd.isnull(row['Customer_Rating'])
        else row['Customer_Rating'],
        axis=1
    )
)

print("\nCustomer Rating:")
print(deliveries['Customer_Rating'].describe())

# ============================================================
# STEP 15 — FAILURE REASON
# ============================================================

deliveries['Failure_Reason'] = (
    deliveries['Failure_Reason']
    .replace(
        [
            'nan',
            'NaN',
            'NA',
            'N/A',
            'None',
            '',
            'null',
            '-'
        ],
        np.nan
    )
)

# Failure reason only belongs to Failed deliveries
deliveries.loc[
    deliveries['Delivery_Status'] != 'Failed',
    'Failure_Reason'
] = np.nan

# Missing reason for failed deliveries
deliveries.loc[
    (
        deliveries['Delivery_Status'] == 'Failed'
    ) &
    (
        deliveries['Failure_Reason'].isnull()
    ),
    'Failure_Reason'
] = 'Reason Not Captured'

print("\nFailure Reason:")
print(deliveries['Failure_Reason'].value_counts())

# ============================================================
# STEP 16 — FIX INVALID DATE SEQUENCES
# ============================================================

invalid_dates = (
    deliveries['Pickup_Date'].notna() &
    deliveries['Actual_Delivery_Date'].notna() &
    (
        deliveries['Actual_Delivery_Date']
        < deliveries['Pickup_Date']
    )
)

print(
    f"\nInvalid date sequences before fixing: "
    f"{invalid_dates.sum()}"
)

# For invalid sequences, derive Actual Delivery Date
# from Pickup Date + Actual Delivery Days
deliveries.loc[
    invalid_dates,
    'Actual_Delivery_Date'
] = (
    deliveries.loc[
        invalid_dates,
        'Pickup_Date'
    ]
    +
    pd.to_timedelta(
        deliveries.loc[
            invalid_dates,
            'Actual_Delivery_Days'
        ],
        unit='D'
    )
)

invalid_dates_after = (
    deliveries['Pickup_Date'].notna() &
    deliveries['Actual_Delivery_Date'].notna() &
    (
        deliveries['Actual_Delivery_Date']
        < deliveries['Pickup_Date']
    )
)

print(
    f"Invalid date sequences after fixing: "
    f"{invalid_dates_after.sum()}"
)

# ============================================================
# STEP 17 — OUTLIER DETECTION ONLY
# DO NOT CAP BUSINESS VALUES
# ============================================================

def detect_outliers(series, col_name):

    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = (
        (series < lower) |
        (series > upper)
    ).sum()

    print(
        f"{col_name}: "
        f"{outliers} statistical outliers | "
        f"bounds [{lower:.2f}, {upper:.2f}]"
    )


print("\n" + "=" * 60)
print("OUTLIER DETECTION")
print("=" * 60)

detect_outliers(
    deliveries['Delivery_Cost'],
    'Delivery_Cost'
)

detect_outliers(
    deliveries['Actual_Delivery_Days'],
    'Actual_Delivery_Days'
)

detect_outliers(
    deliveries['Delay_Days'],
    'Delay_Days'
)

print("No business values capped or modified.")

# ============================================================
# STEP 18 — FEATURE ENGINEERING
# ============================================================

deliveries['Delay_Flag'] = (
    deliveries['Delay_Days'] > 0
).astype(int)

deliveries['On_Time_Flag'] = (
    deliveries['Delay_Days'] <= 0
).astype(int)

deliveries['Early_Delivery_Flag'] = (
    deliveries['Delay_Days'] < 0
).astype(int)

deliveries['Is_Failed'] = (
    deliveries['Delivery_Status'] == 'Failed'
).astype(int)

deliveries['Multiple_Attempt_Flag'] = (
    deliveries['Delivery_Attempts'] > 1
).astype(int)

# Delivery speed
def speed_band(days):

    if days <= 2:
        return 'Express (1-2 days)'

    elif days <= 4:
        return 'Standard (3-4 days)'

    elif days <= 7:
        return 'Slow (5-7 days)'

    else:
        return 'Very Slow (8+ days)'


deliveries['Delivery_Speed_Band'] = (
    deliveries['Actual_Delivery_Days']
    .apply(speed_band)
)

# Cost per day
deliveries['Cost_Per_Day'] = (
    deliveries['Delivery_Cost'] /
    deliveries['Actual_Delivery_Days']
).round(2)

# Delay severity
def delay_severity(days):

    if days < 0:
        return 'Early'

    elif days == 0:
        return 'On Time'

    elif days <= 2:
        return 'Slightly Delayed'

    elif days <= 5:
        return 'Moderately Delayed'

    else:
        return 'Severely Delayed'


deliveries['Delay_Severity'] = (
    deliveries['Delay_Days']
    .apply(delay_severity)
)

# ============================================================
# STEP 19 — PARTNER RELIABILITY SCORE
# SMOOTHED SO IT NEVER BECOMES EXACTLY ZERO
# ============================================================

partner_stats = (
    deliveries
    .groupby('Delivery_Partner')['On_Time_Flag']
    .agg(
        on_time_count='sum',
        total_count='count'
    )
)

# Bayesian smoothing:
# Gives every partner a small non-zero baseline.
# Prior = 1 successful + 1 unsuccessful delivery.
partner_stats['Reliability'] = (
    (
        partner_stats['on_time_count'] + 1
    )
    /
    (
        partner_stats['total_count'] + 2
    )
)

partner_reliability = (
    partner_stats['Reliability'] * 100
)

deliveries['Partner_Reliability_Score'] = (
    deliveries['Delivery_Partner']
    .map(partner_reliability)
    .round(2)
)

print("\nPartner Reliability:")
print(
    deliveries
    .groupby('Delivery_Partner')
    ['Partner_Reliability_Score']
    .first()
    .sort_values(ascending=False)
)

# ============================================================
# STEP 20 — PICKUP DATE FEATURES
# ============================================================

deliveries['Pickup_Month'] = (
    deliveries['Pickup_Date'].dt.month
)

deliveries['Pickup_Quarter'] = (
    deliveries['Pickup_Date'].dt.quarter
)

deliveries['Pickup_DayOfWeek'] = (
    deliveries['Pickup_Date'].dt.day_name()
)

deliveries['Is_Weekend_Pickup'] = (
    deliveries['Pickup_Date']
    .dt.dayofweek
    .isin([5, 6])
    .astype(int)
)

deliveries['Is_Festive_Season'] = (
    deliveries['Pickup_Month']
    .isin([10, 11, 12])
    .astype(int)
)

deliveries['Low_Rating_Flag'] = (
    deliveries['Customer_Rating'] <= 2
).astype(int)

# ============================================================
# STEP 21 — NORMALIZATION
# ============================================================

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler
)

scaler_mm = MinMaxScaler()

deliveries['Delivery_Cost_Normalized'] = (
    scaler_mm.fit_transform(
        deliveries[['Delivery_Cost']]
    )
)

deliveries['Rating_Normalized'] = (
    scaler_mm.fit_transform(
        deliveries[['Customer_Rating']]
    )
)

deliveries['Delay_Days_Normalized'] = (
    scaler_mm.fit_transform(
        deliveries[['Delay_Days']]
    )
)

# Z-score
scaler_z = StandardScaler()

deliveries['Cost_Zscore'] = (
    scaler_z.fit_transform(
        deliveries[['Delivery_Cost']]
    )
)

deliveries['Delay_Zscore'] = (
    scaler_z.fit_transform(
        deliveries[['Delay_Days']]
    )
)

# Log transformations
deliveries['Cost_Log'] = np.log1p(
    deliveries['Delivery_Cost']
)

deliveries['Actual_Days_Log'] = np.log1p(
    deliveries['Actual_Delivery_Days']
)

# ============================================================
# STEP 22 — CATEGORICAL ENCODING
# ============================================================

status_order = {
    'Failed': 0,
    'Returned to Origin': 1,
    'Delayed': 2,
    'In Transit': 3,
    'Out for Delivery': 4,
    'Delivered': 5
}

deliveries['Status_Encoded'] = (
    deliveries['Delivery_Status']
    .map(status_order)
)

severity_order = {
    'Severely Delayed': 0,
    'Moderately Delayed': 1,
    'Slightly Delayed': 2,
    'On Time': 3,
    'Early': 4
}

deliveries['Severity_Encoded'] = (
    deliveries['Delay_Severity']
    .map(severity_order)
)

speed_order = {
    'Very Slow (8+ days)': 0,
    'Slow (5-7 days)': 1,
    'Standard (3-4 days)': 2,
    'Express (1-2 days)': 3
}

deliveries['Speed_Encoded'] = (
    deliveries['Delivery_Speed_Band']
    .map(speed_order)
)

# Partner one-hot encoding
partner_dummies = pd.get_dummies(
    deliveries['Delivery_Partner'],
    prefix='Partner',
    drop_first=True
)

deliveries = pd.concat(
    [deliveries, partner_dummies],
    axis=1
)

# Cyclical month encoding
deliveries['Month_Sin'] = np.sin(
    2 * np.pi *
    deliveries['Pickup_Month'] / 12
)

deliveries['Month_Cos'] = np.cos(
    2 * np.pi *
    deliveries['Pickup_Month'] / 12
)

# ============================================================
# STEP 23 — FINAL SANITY CHECK
# ============================================================

print("\n" + "=" * 65)
print("  FINAL DELIVERY DATA QUALITY REPORT")
print("=" * 65)

print(
    f"Total rows             : {len(deliveries):,}"
)

print(
    f"Total columns          : {len(deliveries.columns)}"
)

print(
    f"Remaining null cells   : "
    f"{deliveries.isnull().sum().sum():,}"
)

print(
    f"Duplicate rows         : "
    f"{deliveries.duplicated().sum()}"
)

print(
    f"Duplicate Delivery_IDs : "
    f"{deliveries['Delivery_ID'].duplicated().sum()}"
)

print(
    f"Duplicate Order_IDs    : "
    f"{deliveries['Order_ID'].duplicated().sum()}"
)

# Foreign key check
invalid_fk_final = (
    ~deliveries['Order_ID']
    .isin(valid_order_ids)
).sum()

print(
    f"Invalid Order_IDs      : "
    f"{invalid_fk_final}"
)

# Numeric checks
print("\nNUMERIC CHECKS")

print(
    "Invalid Expected_Days:",
    (
        ~deliveries['Expected_Days'].between(1, 14)
    ).sum()
)

print(
    "Invalid Actual_Delivery_Days:",
    (
        ~deliveries['Actual_Delivery_Days'].between(1, 30)
    ).sum()
)

print(
    "Invalid Delivery_Attempts:",
    (
        ~deliveries['Delivery_Attempts'].between(1, 5)
    ).sum()
)

print(
    "Invalid Delivery_Cost:",
    (
        deliveries['Delivery_Cost'] <= 0
    ).sum()
)

print(
    "Invalid Customer_Rating:",
    (
        ~deliveries['Customer_Rating'].between(1, 5)
    ).sum()
)

# Date validation
invalid_date_final = (
    deliveries['Pickup_Date'].notna() &
    deliveries['Actual_Delivery_Date'].notna() &
    (
        deliveries['Actual_Delivery_Date']
        < deliveries['Pickup_Date']
    )
).sum()

print(
    "\nActual delivery before pickup:",
    invalid_date_final
)

# Business logic
non_failed_reason = (
    (
        deliveries['Delivery_Status'] != 'Failed'
    ) &
    deliveries['Failure_Reason'].notna()
).sum()

failed_missing_reason = (
    (
        deliveries['Delivery_Status'] == 'Failed'
    ) &
    deliveries['Failure_Reason'].isna()
).sum()

print(
    "Non-failed orders with failure reason:",
    non_failed_reason
)

print(
    "Failed orders missing failure reason:",
    failed_missing_reason
)

# ============================================================
# PERFORMANCE SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("  DELIVERY PERFORMANCE SUMMARY")
print("=" * 65)

print(
    f"On-Time Rate          : "
    f"{deliveries['On_Time_Flag'].mean() * 100:.2f}%"
)

print(
    f"Delayed Rate          : "
    f"{deliveries['Delay_Flag'].mean() * 100:.2f}%"
)

print(
    f"Failed Rate           : "
    f"{deliveries['Is_Failed'].mean() * 100:.2f}%"
)

print(
    f"Early Delivery Rate   : "
    f"{deliveries['Early_Delivery_Flag'].mean() * 100:.2f}%"
)

print(
    f"Average Delay         : "
    f"{deliveries['Delay_Days'].mean():.2f} days"
)

print(
    f"Average Delivery Cost: "
    f"₹{deliveries['Delivery_Cost'].mean():.2f}"
)

print(
    f"Average Rating        : "
    f"{deliveries['Customer_Rating'].mean():.2f}"
)

print("\nPARTNER PERFORMANCE")

print(
    deliveries
    .groupby('Delivery_Partner')
    .agg(
        Orders=('Order_ID', 'count'),
        OnTime_Rate=('On_Time_Flag', 'mean'),
        Avg_Delay=('Delay_Days', 'mean'),
        Avg_Rating=('Customer_Rating', 'mean'),
        Avg_Cost=('Delivery_Cost', 'mean'),
        Reliability=('Partner_Reliability_Score', 'first')
    )
    .assign(
        OnTime_Rate=lambda x:
        x['OnTime_Rate'] * 100
    )
    .round(2)
    .sort_values(
        'Reliability',
        ascending=False
    )
)

# ============================================================
# SAVE FINAL FILES
# ============================================================

deliveries.to_csv(
    'KartZone_Deliveries_Clean.csv',
    index=False
)

essential_cols = [
    'Delivery_ID',
    'Order_ID',
    'Delivery_Partner',
    'Warehouse_City',
    'Pickup_Date',
    'Expected_Days',
    'Actual_Delivery_Days',
    'Delay_Days',
    'Actual_Delivery_Date',
    'Delivery_Status',
    'Delivery_Attempts',
    'Failure_Reason',
    'Delivery_Cost',
    'Customer_Rating',
    'Delay_Flag',
    'On_Time_Flag',
    'Early_Delivery_Flag',
    'Is_Failed',
    'Multiple_Attempt_Flag',
    'Delivery_Speed_Band',
    'Cost_Per_Day',
    'Delay_Severity',
    'Partner_Reliability_Score',
    'Pickup_Month',
    'Pickup_Quarter',
    'Pickup_DayOfWeek',
    'Is_Weekend_Pickup',
    'Is_Festive_Season',
    'Low_Rating_Flag'
]

deliveries[essential_cols].to_csv(
    'KartZone_Deliveries_Final.csv',
    index=False
)

print("\n" + "=" * 65)
print("  DELIVERY CLEANING COMPLETE")
print("=" * 65)

print(
    "Saved → KartZone_Deliveries_Clean.csv ✓"
)

print(
    "Saved → KartZone_Deliveries_Final.csv ✓"
)



  KARTZONE INDIA — DELIVERY DATA CLEANING

Raw rows loaded: 9,713
Rows after structural cleaning: 9,598
Invalid Order_ID format: 0
Invalid Order_IDs: 0
Rows after foreign-key validation: 9,598

Delivery Partner:
Delivery_Partner
Delhivery           2514
Ekart               1828
Blue Dart           1652
DTDC                1391
XpressBees           920
Unknown              663
Amazon Logistics     630
Name: count, dtype: int64

Warehouse City:
Warehouse_City
Hyderabad Hub    2440
Delhi Hub        1481
Pune Hub         1447
Mumbai Hub       1436
Chennai Hub      1402
Bangalore Hub    1392
Name: count, dtype: int64

Date types:
Pickup_Date             datetime64[ns]
Actual_Delivery_Date    datetime64[ns]
dtype: object

Expected_Days:
count    9598.000000
mean        4.990936
std         1.413116
min         3.000000
25%         4.000000
50%         5.000000
75%         6.000000
max         7.000000
Name: Expected_Days, dtype: float64

Missing Actual_Delivery_Days before fill: 1663
Missing

In [2]:
# ============================================================
# FINAL PARTNER RELIABILITY SCORE
# ============================================================

# Reliability is based primarily on the percentage of
# deliveries completed on time or early.

partner_ontime_rate = (
    deliveries.groupby('Delivery_Partner')['On_Time_Flag']
    .transform('mean')
)

# Convert to percentage
deliveries['Partner_Reliability_Score'] = (
    partner_ontime_rate * 100
)

# Apply a small reporting floor so no partner has a literal 0
# while preserving the relative performance ranking.
deliveries['Partner_Reliability_Score'] = (
    deliveries['Partner_Reliability_Score']
    .clip(lower=5, upper=100)
    .round(2)
)

print("\nPARTNER RELIABILITY SCORE")
print(
    deliveries.groupby('Delivery_Partner')
    ['Partner_Reliability_Score']
    .first()
    .sort_values(ascending=False)
)

print("\n============================================================")
print("PARTNER RELIABILITY SCORE COMPLETE")
print("============================================================")


PARTNER RELIABILITY SCORE
Delivery_Partner
Amazon Logistics    38.57
Delhivery           32.06
Unknown             28.21
Ekart               24.89
Blue Dart           13.26
XpressBees           8.37
DTDC                 5.82
Name: Partner_Reliability_Score, dtype: float64

PARTNER RELIABILITY SCORE COMPLETE


In [3]:
# ============================================================
# FINAL DELIVERY VALIDATION
# ============================================================

print("\n" + "="*60)
print("  KARTZONE — FINAL DELIVERY DATA QUALITY CHECK")
print("="*60)

print(f"\nRows                  : {len(deliveries):,}")
print(f"Columns               : {len(deliveries.columns):,}")
print(f"Blank rows            : {deliveries.isnull().all(axis=1).sum()}")
print(f"Exact duplicates      : {deliveries.duplicated().sum()}")
print(f"Duplicate Delivery_IDs: {deliveries['Delivery_ID'].duplicated().sum()}")
print(f"Duplicate Order_IDs   : {deliveries['Order_ID'].duplicated().sum()}")

# ------------------------------------------------------------
# NULL CHECK
# ------------------------------------------------------------

print("\nNULL CHECK")
print(f"Remaining null cells : {deliveries.isnull().sum().sum():,}")

# ------------------------------------------------------------
# NUMERIC VALIDATION
# ------------------------------------------------------------

invalid_expected = (
    (deliveries['Expected_Days'] < 1) |
    (deliveries['Expected_Days'] > 14)
).sum()

invalid_actual = (
    (deliveries['Actual_Delivery_Days'] < 1) |
    (deliveries['Actual_Delivery_Days'] > 30)
).sum()

invalid_attempts = (
    (deliveries['Delivery_Attempts'] < 1) |
    (deliveries['Delivery_Attempts'] > 5)
).sum()

invalid_cost = (
    deliveries['Delivery_Cost'] <= 0
).sum()

invalid_rating = (
    (deliveries['Customer_Rating'] < 1) |
    (deliveries['Customer_Rating'] > 5)
).sum()

print("\nNUMERIC VALIDATION")
print(f"Invalid Expected_Days       : {invalid_expected}")
print(f"Invalid Actual_Delivery_Days: {invalid_actual}")
print(f"Invalid Delivery_Attempts   : {invalid_attempts}")
print(f"Invalid Delivery_Cost       : {invalid_cost}")
print(f"Invalid Customer_Rating     : {invalid_rating}")

# ------------------------------------------------------------
# DATE VALIDATION
# ------------------------------------------------------------

invalid_dates = (
    deliveries['Actual_Delivery_Date'] <
    deliveries['Pickup_Date']
).sum()

print("\nDATE VALIDATION")
print(f"Actual delivery before pickup: {invalid_dates}")

# ------------------------------------------------------------
# BUSINESS LOGIC VALIDATION
# ------------------------------------------------------------

expected_delay = (
    deliveries['Actual_Delivery_Days'] -
    deliveries['Expected_Days']
)

delay_mismatch = (
    deliveries['Delay_Days'] != expected_delay
).sum()

non_failed_reason = (
    (deliveries['Delivery_Status'] != 'Failed') &
    deliveries['Failure_Reason'].notna()
).sum()

failed_missing_reason = (
    (deliveries['Delivery_Status'] == 'Failed') &
    deliveries['Failure_Reason'].isna()
).sum()

print("\nBUSINESS LOGIC")
print(f"Delay calculation mismatches : {delay_mismatch}")
print(f"Non-failed with Failure_Reason: {non_failed_reason}")
print(f"Failed without Failure_Reason : {failed_missing_reason}")

# ------------------------------------------------------------
# PARTNER RELIABILITY
# ------------------------------------------------------------

partner_summary = (
    deliveries.groupby('Delivery_Partner')
    .agg(
        Orders=('Delivery_ID', 'count'),
        OnTime_Rate=('On_Time_Flag', 'mean'),
        Avg_Delay=('Delay_Days', 'mean'),
        Avg_Rating=('Customer_Rating', 'mean'),
        Avg_Cost=('Delivery_Cost', 'mean'),
        Reliability=('Partner_Reliability_Score', 'first')
    )
)

partner_summary['OnTime_Rate'] = (
    partner_summary['OnTime_Rate'] * 100
)

partner_summary = partner_summary.round(2)

print("\nPARTNER PERFORMANCE")
print(
    partner_summary
    .sort_values('Reliability', ascending=False)
)

# ------------------------------------------------------------
# DELIVERY PERFORMANCE
# ------------------------------------------------------------

on_time_rate = deliveries['On_Time_Flag'].mean() * 100
delayed_rate = deliveries['Delay_Flag'].mean() * 100
failed_rate = deliveries['Is_Failed'].mean() * 100
early_rate = deliveries['Early_Delivery_Flag'].mean() * 100
avg_delay = deliveries['Delay_Days'].mean()
avg_cost = deliveries['Delivery_Cost'].mean()
avg_rating = deliveries['Customer_Rating'].mean()

print("\nDELIVERY PERFORMANCE")
print(f"On-Time Rate       : {on_time_rate:.2f}%")
print(f"Delayed Rate       : {delayed_rate:.2f}%")
print(f"Failed Rate        : {failed_rate:.2f}%")
print(f"Early Delivery Rate: {early_rate:.2f}%")
print(f"Average Delay      : {avg_delay:.2f} days")
print(f"Average Cost       : ₹{avg_cost:.2f}")
print(f"Average Rating     : {avg_rating:.2f}")

print("\n" + "="*60)
print("  FINAL DELIVERY VALIDATION COMPLETE")
print("="*60)


  KARTZONE — FINAL DELIVERY DATA QUALITY CHECK

Rows                  : 9,598
Columns               : 47
Blank rows            : 0
Exact duplicates      : 0
Duplicate Delivery_IDs: 0
Duplicate Order_IDs   : 0

NULL CHECK
Remaining null cells : 21,309

NUMERIC VALIDATION
Invalid Expected_Days       : 0
Invalid Actual_Delivery_Days: 0
Invalid Delivery_Attempts   : 0
Invalid Delivery_Cost       : 0
Invalid Customer_Rating     : 0

DATE VALIDATION
Actual delivery before pickup: 0

BUSINESS LOGIC
Delay calculation mismatches : 0
Non-failed with Failure_Reason: 0
Failed without Failure_Reason : 0

PARTNER PERFORMANCE
                  Orders  OnTime_Rate  Avg_Delay  Avg_Rating  Avg_Cost  \
Delivery_Partner                                                         
Amazon Logistics     630        38.57       0.63        3.85     97.41   
Delhivery           2514        32.06       1.03        3.72    103.27   
Unknown              663        28.21       1.60        3.53     97.48   
Ekart     

In [4]:
# ============================================================
# SAVE FINAL DELIVERY TABLE
# ============================================================

deliveries.to_csv(
    'KartZone_Deliveries_Clean.csv',
    index=False
)

deliveries.to_csv(
    'KartZone_Deliveries_Final.csv',
    index=False
)

print("\n============================================================")
print("FINAL DELIVERY FILES SAVED")
print("============================================================")
print("Saved → KartZone_Deliveries_Clean.csv ✓")
print("Saved → KartZone_Deliveries_Final.csv ✓")
print(f"Rows    : {len(deliveries):,}")
print(f"Columns : {len(deliveries.columns):,}")


FINAL DELIVERY FILES SAVED
Saved → KartZone_Deliveries_Clean.csv ✓
Saved → KartZone_Deliveries_Final.csv ✓
Rows    : 9,598
Columns : 47
